In [1]:
import surprise
import pandas as pd
import numpy as np
import os
os.chdir("C:/Users/PGCP-AI/ML/MachineLearning/Cases_Rec_Sys/ml-100k")

In [2]:
rating = pd.read_csv('u.data',sep='\t',names = ['uid','Iid','rating','Timestamp'])
rating

,uid,Iid,rating,Timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596
...,...,...,...,...
99995,880,476,3,880175444
99996,716,204,5,879795543
99997,276,1090,1,874795795
99998,13,225,2,882399156


In [3]:
rating.drop('Timestamp',axis=1,inplace=True)

In [4]:
lowest_rating = rating['rating'].min()
highest_rating = rating['rating'].max()
lowest_rating,highest_rating

(1, 5)

In [5]:
reader = surprise.Reader(rating_scale=(lowest_rating,highest_rating))
data = surprise.Dataset.load_from_df(df=rating,reader=reader)

In [6]:
similarity_options = {'name':'cosine','user_based':True}
algo = surprise.KNNBasic(sim_options=similarity_options)
output = algo.fit(data.build_full_trainset()) 

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [7]:
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold
param_grid ={'k':[20,30,50,70,90],'user_based':[True,False]}
kfold= KFold(n_splits=5,shuffle=True,random_state=26)
# gs=GridSearchCV(surprise.KNNBasic,param_grid,measures=['rmse','mae'],cv=kfold)
gs=GridSearchCV(surprise.KNNWithZScore,param_grid,measures=['rmse','mae'],cv=kfold)
gs.fit(data)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

In [8]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.9499384745978843
{'k': 70, 'user_based': True}


In [9]:
user = 50
u_iid = rating[rating['uid']==user]['Iid'].unique()
iids = rating['Iid'].unique()
print('List of items rated by user:',u_iid)
print('No. of items rated by user {0} : {1} '.format(user,len(u_iid)))
iids_to_predict = np.setdiff1d(iids,u_iid)
print("Items not rated by the user or those items for which the expected ratings are to be predicted",iids_to_predict)

List of items rated by user: [ 246  823  253  475 1084  286    9  125  123  325  508  288  319  324
  276 1008 1010  268  544   15  327  124  547  100]
No. of items rated by user 50 : 24 
Items not rated by the user or those items for which the expected ratings are to be predicted [   1    2    3 ... 1680 1681 1682]


In [10]:
testset = [[user,iid,0.] for iid in iids_to_predict]
predictions = algo.test(testset)
exp_ratings = [ (predictions[i].iid, predictions[i].est) for i in range(0, len(predictions))]
df  = pd.DataFrame(exp_ratings, columns = ["iid", "est_rating"])
df.sort_values('est_rating',ascending=False)

,iid,est_rating
1475,1500,5.0
1442,1467,5.0
1164,1189,5.0
1097,1122,5.0
1628,1653,5.0
...,...,...
1552,1577,1.0
1551,1576,1.0
1550,1575,1.0
1549,1574,1.0


In [11]:
movies = pd.read_csv('movies_list.csv',encoding='latin-1')

In [12]:
user = 1000
u_iid = rating[rating['uid']==user]['Iid'].unique()
iids = rating['Iid'].unique()
iids_to_predict = np.setdiff1d(iids,u_iid)
testset = [[user,iid,0.] for iid in iids_to_predict]
predictions = algo.test(testset)
exp_ratings = [ (predictions[i].iid, predictions[i].est) for i in range(0, len(predictions))]
df  = pd.DataFrame(exp_ratings, columns = ["iid", "est_rating"])
df = df.merge(movies[['movie id ', ' movie title ', ' release date ']], how = 'left', left_on = 'iid', right_on = 'movie id ')
df.sort_values(by = 'est_rating', ascending = False).head(10)[['movie id ', ' movie title ', ' release date ']]

,movie id,movie title,release date
0,1,Toy Story (1995),01-Jan-95
1104,1105,Firestorm (1998),09-Jan-98
1128,1129,Chungking Express (1994),16-Feb-96
1127,1128,Heidi Fleiss: Hollywood Madam (1995),09-Feb-96
1126,1127,"Truman Show, The (1998)",01-Jan-98
1125,1126,"Old Man and the Sea, The (1958)",01-Jan-58
1124,1125,"Innocents, The (1961)",01-Jan-61
1123,1124,"Farewell to Arms, A (1932)",01-Jan-32
1122,1123,"Last Time I Saw Paris, The (1954)",01-Jan-54
1121,1122,They Made Me a Criminal (1939),01-Jan-39


In [13]:
from ipywidgets import interact
import ipywidgets as widgets
from IPython.display import display,clear_output

In [14]:
def predict_user_review(user):
    u_iid = rating[rating['uid']==user]['Iid'].unique()
    iids_to_predict = np.setdiff1d(iids,u_iid)
    testset = [[user,iid,0.] for iid in iids_to_predict]
    predictions = algo.test(testset)
    exp_ratings = [ (predictions[i].iid, predictions[i].est) for i in range(0, len(predictions))]
    df  = pd.DataFrame(exp_ratings, columns = ["iid", "est_rating"])
    df = df.merge(movies[['movie id ', ' movie title ', ' release date ']], how = 'left', left_on = 'iid', right_on = 'movie id ')
    return df.sort_values(by = 'est_rating', ascending = False).head(10)[['movie id ', ' movie title ', ' release date ']]

In [15]:
rating['uid'].min(),rating['uid'].max()
rating['Iid'].min(),rating['Iid'].max()


(1, 1682)

In [16]:
unique_users = rating['uid'].unique()

In [19]:
output = interact(predict_user_review, user = iids)
display(output)

,movie id,movie title,release date
1578,1599,Someone Else's America (1995),10-May-96
1105,1122,They Made Me a Criminal (1939),01-Jan-39
1274,1293,Star Kid (1997),16-Jan-98
1632,1653,Entertaining Angels: The Dorothy Day Story (1996),27-Sep-96
1446,1467,"Saint of Fort Washington, The (1993)",01-Jan-93
1635,1656,Little City (1998),20-Feb-98
1515,1536,Aiqing wansui (1994),22-Jul-96
1182,1201,Marlene Dietrich: Shadow and Light (1996),02-Apr-96
799,814,"Great Day in Harlem, A (1994)",01-Jan-94
1479,1500,Santa with Muscles (1996),08-Nov-96


interactive(children=(Dropdown(description='user', options=(242, 302, 377, 51, 346, 474, 265, 465, 451, 86, 25…

<function __main__.predict_user_review(user)>